# 10_run_evaluation

無加工正例データ、誤植付き正例データ、負例データに対して全件評価を実行し、テキスト集計とグラフで結果を確認する。

## 評価対象

この notebook では以下の 3 データセットを評価する。

1. `base_positive_examples.json`
2. `positive_examples.json`
3. `negative_examples.json`

BM25 は別枠評価とし、`RC / CC / MC` は方式別 1 位候補への閾値判定を行う。

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# グラフ表示 backend を設定する
try:
    get_ipython().run_line_magic("matplotlib", "qt")
    print("matplotlib backend: qt (別ウィンドウ表示)")
except Exception as exc:
    get_ipython().run_line_magic("matplotlib", "inline")
    print("matplotlib backend: inline (notebook 内表示)")
    print(f"qt backend は使えませんでした: {exc}")


In [ ]:
# 設定値・評価関数・描画ライブラリを読み込む
from pprint import pprint

import matplotlib.pyplot as plt

from tanigawa_shoshi.config import (
    BASE_POSITIVE_EXAMPLES_PATH,
    NEGATIVE_EXAMPLES_PATH,
    POSITIVE_EXAMPLES_PATH,
)
from tanigawa_shoshi.evaluation import DEFAULT_THRESHOLDS, evaluate_dataset
from tanigawa_shoshi.evaluation_data import load_examples

plt.rcParams["font.family"] = ["Hiragino Sans", "Yu Gothic", "IPAexGothic", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
# 評価設定を指定する
rows = 100
thresholds = DEFAULT_THRESHOLDS.copy()

print(f"rows: {rows}")
print("thresholds")
for score_name in ["rc", "cc", "mc"]:
    print(f"  {score_name}: {thresholds[score_name]}")


In [ ]:
# 評価対象データを読み込む
base_positive_examples = load_examples(BASE_POSITIVE_EXAMPLES_PATH)
positive_examples = load_examples(POSITIVE_EXAMPLES_PATH)
negative_examples = load_examples(NEGATIVE_EXAMPLES_PATH)

print(f"base positive count: {len(base_positive_examples)}")
print(f"positive count: {len(positive_examples)}")
print(f"negative count: {len(negative_examples)}")


In [ ]:
# 3 データセットを全件評価する
#　約二分
base_positive_evaluation = evaluate_dataset(
    base_positive_examples,
    thresholds,
    rows=rows,
    mode="positive",
)
positive_evaluation = evaluate_dataset(
    positive_examples,
    thresholds,
    rows=rows,
    mode="positive",
)
negative_evaluation = evaluate_dataset(
    negative_examples,
    thresholds,
    rows=rows,
    mode="negative",
)

print("evaluation finished")


In [ ]:
# テキスト集計を表形式で表示する補助関数
OUTCOME_LABELS = {
    "detected_count": "正常検出",
    "missed_count": "検出漏れ",
    "false_positive_count": "誤検出",
    "true_negative_count": "正しく非同定",
}
METHOD_LABELS = {
    "bm25": "BM25",
    "rc": "RC",
    "cc": "CC",
    "mc": "MC",
}
DATASET_LABELS = {
    "base_positive": "無加工正例",
    "positive": "誤植付き正例",
    "negative": "負例",
}
METHOD_ORDER = ["bm25", "rc", "cc", "mc"]


def format_count_ratio(value, total_count):
    ratio = (value / total_count * 100) if total_count else 0.0
    return f"{value:,} ({ratio:.2f}%)"


def print_text_table(title, headers, rows):
    widths = [len(header) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    def render_row(row_values):
        return " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(row_values))

    separator = "-+-".join("-" * width for width in widths)

    print("=" * 100)
    print(title)
    print(render_row(headers))
    print(separator)
    for row in rows:
        print(render_row(row))


def build_positive_table_rows(summary):
    total_count = summary["input_count"]
    rows = []
    for method_name in METHOD_ORDER:
        method_summary = summary["methods"].get(method_name, {})
        rows.append(
            [
                METHOD_LABELS[method_name],
                format_count_ratio(method_summary.get("detected_count", 0), total_count),
                format_count_ratio(method_summary.get("missed_count", 0), total_count),
                format_count_ratio(method_summary.get("false_positive_count", 0), total_count),
            ]
        )
    return rows


def build_negative_table_rows(summary):
    total_count = summary["input_count"]
    rows = []
    for method_name in METHOD_ORDER:
        method_summary = summary["methods"].get(method_name, {})
        rows.append(
            [
                METHOD_LABELS[method_name],
                format_count_ratio(method_summary.get("true_negative_count", 0), total_count),
                format_count_ratio(method_summary.get("false_positive_count", 0), total_count),
            ]
        )
    return rows


def print_positive_summary(dataset_name, evaluation_result):
    print_text_table(
        f"{DATASET_LABELS.get(dataset_name, dataset_name)} の評価結果",
        ["スコアリング", "正常検出", "検出漏れ", "誤検出"],
        build_positive_table_rows(evaluation_result["summary"]),
    )


def print_negative_summary(dataset_name, evaluation_result):
    print_text_table(
        f"{DATASET_LABELS.get(dataset_name, dataset_name)} の評価結果",
        ["スコアリング", "正しく非同定", "誤検出"],
        build_negative_table_rows(evaluation_result["summary"]),
    )

In [ ]:
# 集計結果をテキストで表形式表示する
print_positive_summary("base_positive", base_positive_evaluation)
print()
print_positive_summary("positive", positive_evaluation)
print()
print_negative_summary("negative", negative_evaluation)

In [ ]:
# グラフ描画用の補助関数
METHOD_ORDER = ["bm25", "rc", "cc", "mc"]
POSITIVE_OUTCOME_KEYS = ["detected_count", "missed_count", "false_positive_count"]
NEGATIVE_OUTCOME_KEYS = ["true_negative_count", "false_positive_count"]
POSITIVE_COLORS = {
    "detected_count": "#4daf4a",
    "missed_count": "#ff7f00",
    "false_positive_count": "#e41a1c",
}
NEGATIVE_COLORS = {
    "true_negative_count": "#377eb8",
    "false_positive_count": "#e41a1c",
}

def extract_method_counts(summary, outcome_keys):
    method_counts = []
    for method_name in METHOD_ORDER:
        method_summary = summary["methods"].get(method_name, {})
        method_counts.append([method_summary.get(key, 0) for key in outcome_keys])
    return method_counts

def plot_stacked_counts(ax, dataset_name, summary, outcome_keys, colors):
    method_counts = extract_method_counts(summary, outcome_keys)
    input_count = summary["input_count"]
    bottoms = [0] * len(METHOD_ORDER)
    for outcome_index, outcome_key in enumerate(outcome_keys):
        values = [counts[outcome_index] for counts in method_counts]
        labels = [OUTCOME_LABELS[outcome_key]] * len(values)
        bars = ax.bar(
            [METHOD_LABELS[m] for m in METHOD_ORDER],
            values,
            bottom=bottoms,
            color=colors[outcome_key],
            label=OUTCOME_LABELS[outcome_key],
        )
        for bar, value, bottom in zip(bars, values, bottoms):
            if value <= 0:
                continue
            ratio = (value / input_count * 100) if input_count else 0.0
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom + value / 2,
                f"{value}\n({ratio:.1f}%)",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
            )
        bottoms = [bottom + value for bottom, value in zip(bottoms, values)]

    ax.set_title(DATASET_LABELS.get(dataset_name, dataset_name))
    ax.set_ylabel("件数")
    ax.set_xlabel("評価方式")
    ax.grid(axis="y", alpha=0.3)


In [ ]:
# 無加工正例・正例・負例の評価結果を別ウィンドウの stacked bar chart で表示する
fig, axes = plt.subplots(3, 1, figsize=(12, 18), constrained_layout=True)

plot_stacked_counts(
    axes[0],
    "base_positive",
    base_positive_evaluation["summary"],
    POSITIVE_OUTCOME_KEYS,
    POSITIVE_COLORS,
)
plot_stacked_counts(
    axes[1],
    "positive",
    positive_evaluation["summary"],
    POSITIVE_OUTCOME_KEYS,
    POSITIVE_COLORS,
)
plot_stacked_counts(
    axes[2],
    "negative",
    negative_evaluation["summary"],
    NEGATIVE_OUTCOME_KEYS,
    NEGATIVE_COLORS,
)

positive_handles, positive_labels = axes[0].get_legend_handles_labels()
axes[0].legend(positive_handles, positive_labels, loc="upper right")
negative_handles, negative_labels = axes[2].get_legend_handles_labels()
axes[2].legend(negative_handles, negative_labels, loc="upper right")

fig.suptitle("評価結果の集計", fontsize=16)
plt.show()
